In [1]:
import pandas as pd
import numpy as np
from geopy.distance import geodesic
import numpy as np
import os
import warnings

warnings.filterwarnings("ignore")

In [2]:
file_name = 'COTA_KINGElvis.xlsx'

elvis_df=pd.read_excel(file_name,sheet_name='Elvis_Review')

traditional_df=pd.read_csv('COTA_Traditional_Transfer_Flags.csv')

od_df=pd.read_csv('COTA_OD_Distance_Checks.csv')

transfer_df=pd.read_csv('COTA_Distance_Transfer_Flags(001).csv')

recovery_df=pd.read_excel('COTA_survey_recovery_2023-12-05.xlsx', sheet_name='_(F0) SURVEY RECOVERY')

In [3]:
elvis_df=elvis_df[elvis_df['Final_Usage'].str.lower()=='use']

In [4]:
elvis_df.head()

,Elvis_Date,elvis_id,1st Cleaner,FINAL_REVIEWER,Final_Usage,REASON FOR REMOVAL,REASON FOR REMOVAL [Other],route_match_flag,distance_flag,SUPERVISOR_COMMENT,...,2X_REVIEW_CHECK.1,2x_REVIEWED_BY,2x_REVIEWED_FLAG,ADMIN_APPROVED,SURVEY_RECOVERY,SURVEY_RECOVERY_REVIEWED_BY,Recovery Check,2x_REVIEWED_BY.1,2x_REVIEWED_FLAG.1,ADMIN_APPROVED.1
16,2023-11-03,5632,HereAPI,"T A Divya, Regina",Use,NaN,,Elvis_Review,Elvis_Review,NaN,...,NaT,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN
17,2023-11-03,5634,HereAPI,"T A Divya, Regina",Use,NaN,,Elvis_Review,Elvis_Review,NaN,...,NaT,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN
18,2023-11-03,5636,HereAPI,"T A Divya, Regina",Use,NaN,,Elvis_Review,Elvis_Review,NaN,...,NaT,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN
24,2023-11-03,5646,HereAPI,"T A Divya, Brian, Regina",Use,NaN,,Elvis_Review,Elvis_Review,"o= home = E Main St & Wilson Ave, Columbus, OH...",...,NaT,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN
25,2023-11-03,5647,HereAPI,"T A Divya, Regina",Use,NaN,,Elvis_Review,Elvis_Review,NaN,...,NaT,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN


In [5]:
traditional_df.head()

,id,TRIP_FIRST_ROUTECode,TRIP_SECOND_ROUTECode,TRIP_THIRD_ROUTECode,TRIP_FOURTH_ROUTECode,ROUTE_SURVEYEDCode,TRIP_NEXT_ROUTECode,TRIP_AFTER_ROUTECode,TRIP_3RD_ROUTECode,TRIP_LAST4TH_RTECode,...,Transfer5,Check5.GoodTransfer,Transfer6,Check6.GoodTransfer,Transfer7,Check7.GoodTransfer,Transfer8,Check8.GoodTransfer,real flags,Checkall.GoodTransfer
0,5552,NaN,NaN,NaN,NaN,BCT_1_109,BCT_1_28,BCT_1_18,NaN,NaN,...,NaN,0,NaN,0,NaN,0,NaN,0,0,1
1,6154,BCT_1_19,NaN,NaN,NaN,BCT_1_22,NaN,NaN,NaN,NaN,...,NaN,0,NaN,0,NaN,0,NaN,0,0,1
2,7121,NaN,NaN,NaN,NaN,BCT_1_01,BCT_1_01,NaN,NaN,NaN,...,NaN,0,NaN,0,NaN,0,NaN,0,0,1
3,7403,NaN,NaN,NaN,NaN,BCT_1_18,BCT_1_02,BCT_1_55,BCT_1_19,NaN,...,NaN,0,NaN,0,NaN,0,NaN,0,0,1
4,7991,BCT_1_101,NaN,NaN,NaN,BCT_1_36,NaN,NaN,NaN,NaN,...,NaN,0,NaN,0,NaN,0,NaN,0,0,1


In [6]:
od_df.head()

,id,Final_Direction_Code,Final_Direction,HOME_ADDRESS_PLACE,HOME_ADDRESS_ADDR,HOME_ADDRESS_CITY,HOME_ADDRESS_STATE,HOME_ADDRESS_ZIP,HOME_ADDRESS_LAT,HOME_ADDRESS_LONG,...,O_B_Dist_Check3,A_D_Dist_Check1,A_D_Dist_Check2,A_D_Dist_Check3,O_D_Dist_Check1,O_D_Dist_Check2,O_D_Dist_Check3,B_A_Dist_Check1,B_A_Dist_Check2,SUM_ALL_CHECKS
0,5454,BCT_1_109_01,109 - 95 Express Pembroke Pines/Miramar [SOUTH...,801 SW 143rd Ave,801 SW 143rd Ave,Pembroke Pines,Florida,33027,26.002003,-80.333842,...,0,0,0,0,0,0,1,0,0,1
1,5493,BCT_1_106_01,106 - 95 Express Miramar [SOUTHBOUND],-,Miramar Pkwy & Flamingo Rd,Miramar,Florida,33025,25.978670,-80.310638,...,0,0,0,0,0,0,0,0,0,1
2,5523,BCT_1_109_01,109 - 95 Express Pembroke Pines/Miramar [SOUTH...,920 SW 147th Ave,920 SW 147th Ave,Pembroke Pines,Florida,33027,26.000886,-80.343337,...,0,0,0,0,0,0,0,0,0,1
3,5565,BCT_1_109_01,109 - 95 Express Pembroke Pines/Miramar [SOUTH...,-,11645 SW 13th Dr,Pembroke Pines,Florida,33025,25.996528,-80.302298,...,0,0,0,0,0,1,0,0,0,1
4,5803,BCT_1_106_00,106 - 95 Express Miramar [NORTHBOUND],3308 SW 175th Terrace,3308 SW 175th Terrace,Miramar,Florida,33029,25.977485,-80.380429,...,0,0,0,0,0,0,0,0,0,1


In [7]:
transfer_df.head()

,id,ROUTE_SURVEYEDCode,PREV_TRANSFERSCode,PREV_TRANSFERS,TRIP_FIRST_ROUTECode,TRIP_SECOND_ROUTECode,TRIP_THIRD_ROUTECode,TRIP_FOURTH_ROUTECode,NEXT_TRANSFERSCode,NEXT_TRANSFERS,...,Final_Usage_y,Transfer1_Distance,Transfer2_Distance,Transfer3_Distance,Transfer4_Distance,Transfer5_Distance,Transfer6_Distance,Transfer7_Distance,Transfer8_Distance,Flaged
0,5552,BCT_1_109_01,0.0,(0) None,NaN,NaN,NaN,NaN,2.0,(2) Two,...,Use,NaN,NaN,NaN,NaN,0.561182,0.088257,NaN,NaN,1
1,6154,BCT_1_22_00,1.0,(1) One,BCT_1_19,NaN,NaN,NaN,0.0,(0) None,...,Use,1.496056,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
2,6211,BCT_1_09_01,0.0,(0) None,NaN,NaN,NaN,NaN,1.0,(1) One,...,Use,NaN,NaN,NaN,NaN,0.499479,NaN,NaN,NaN,1
3,6424,BCT_1_22_01,0.0,(0) None,NaN,NaN,NaN,NaN,1.0,(1) One,...,Use,NaN,NaN,NaN,NaN,0.472184,NaN,NaN,NaN,1
4,6550,BCT_1_09_01,1.0,(1) One,BCT_1_441,NaN,NaN,NaN,0.0,(0) None,...,Use,0.500458,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1


In [8]:
elvis_df.shape

(6620, 31)

In [9]:
traditional_df.shape

(108, 30)

In [10]:
transfer_df.shape

(247, 71)

In [11]:
od_df.shape

(237, 77)

In [12]:
merged_df = pd.merge(elvis_df, traditional_df['id'], on='id', how='left', indicator=True)

In [13]:
# Create a new column 'Traditional_Check' based on the merge indicator
merged_df['Traditional_Check'] = (merged_df['_merge'] == 'both').astype(int)

# Drop the indicator column and display the resulting DataFrame
merged_df = merged_df.drop(columns=['_merge'])

In [14]:
merged_df[merged_df['Traditional_Check']==1]

,Elvis_Date,elvis_id,1st Cleaner,FINAL_REVIEWER,Final_Usage,REASON FOR REMOVAL,REASON FOR REMOVAL [Other],route_match_flag,distance_flag,SUPERVISOR_COMMENT,...,DESTIN_TRANSPORT,_INTRV_NOTE,ELVIS_STATUS,ELVIS_COMMENT,ROUTE_STATUS,Stops_Status,Test_Status,ROUTE_ONLY,Unnamed: 30,Traditional_Check
59,2023-09-22 00:00:00,5552,HereAPI,Kirubakaran,Use,,,Elvis_Review,Elvis_Review,NaN,...,Walk,NaN,Delete,NaN,Elvis_Review,Elvis_Review,Tested,NaN,NaN,1
210,2023-09-22 00:00:00,6154,HereAPI,Vaishnavi,Use,,,Elvis_Review,Elvis_Review,"D=12801 West Sunrise Boulevard, Sunrise, FL, USA",...,Walk,NaN,Fixable - needs additional review,"D=12801 West Sunrise Boulevard, Sunrise, FL, USA",Elvis_Review,Elvis_Review,Tested,NaN,NaN,1
455,2023-09-30 00:00:00,7121,HereAPI,muskan,Use,,,Elvis_Review,Elvis_Review,NaN,...,Walk,NaN,NaN,NaN,Elvis_Review,Elvis_Review,Tested,NaN,NaN,1
530,2023-09-30 00:00:00,7403,HereAPI,"Priyanka, Brian",Use,NaN,,Elvis_Review,Elvis_Review,Add A transfers Routes 19 and 83,...,Walk,NaN,Delete,NaN,Elvis_Review,Elvis_Review,Tested,NaN,NaN,1
719,2023-09-30 00:00:00,7991,HereAPI,Raja,Use,,,Elvis_Review,Elvis_Review,NaN,...,Walk,NaN,NaN,NaN,Elvis_Review,Elvis_Review,Tested,NaN,NaN,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6159,2023-10-28 00:00:00,21262,HereAPI,Saurav,Use,NaN,NaN,Elvis_Review,Elvis_Review,ADJUST A AND TRANSFER POINTS,...,Walk,NaN,Fixable - needs additional review,ADJUST A AND TRANSFER POINTS,Review,Review,Tested,NaN,NaN,1
6253,2023-10-28 00:00:00,21435,HereAPI,Sakshi,Use,NaN,NaN,Elvis_Review,Elvis_Review,NaN,...,Walk,NaN,NaN,NaN,Review,Review,Tested,NaN,NaN,1
6325,2023-10-28 00:00:00,21574,HereAPI,Himanshu,Use,NaN,NaN,Elvis_Review,Elvis_Review,NaN,...,Walk,NaN,NaN,NaN,Review,Review,Tested,NaN,NaN,1
6335,2023-10-28 00:00:00,21593,HereAPI,Himanshu,Use,NaN,NaN,Elvis_Review,Elvis_Review,"D=3740 West Hillsboro Boulevard, Deerfield Bea...",...,Walk,NaN,Fixable - needs additional review,"D=3740 West Hillsboro Boulevard, Deerfield Bea...",Review,Review,Tested,NaN,NaN,1


In [15]:
merged_df = pd.merge(merged_df, od_df['id'], on='id', how='left', indicator=True)

# Create a new column 'Traditional_Check' based on the merge indicator
merged_df['OD_Distance_Check'] = (merged_df['_merge'] == 'both').astype(int)

# Drop the indicator column and display the resulting DataFrame
merged_df = merged_df.drop(columns=['_merge'])

In [16]:
merged_df[merged_df['OD_Distance_Check']==1]

,Elvis_Date,elvis_id,1st Cleaner,FINAL_REVIEWER,Final_Usage,REASON FOR REMOVAL,REASON FOR REMOVAL [Other],route_match_flag,distance_flag,SUPERVISOR_COMMENT,...,_INTRV_NOTE,ELVIS_STATUS,ELVIS_COMMENT,ROUTE_STATUS,Stops_Status,Test_Status,ROUTE_ONLY,Unnamed: 30,Traditional_Check,OD_Distance_Check
18,2023-09-22 00:00:00,5454,HereAPI,Kirubakaran,Use,,,Elvis_Review,Elvis_Review,"1015 North America Way, Miami, FL, USA, REMOVE...",...,NaN,Fixable - needs additional review,"1015 North America Way, Miami, FL, USA, REMOVE...",Elvis_Review,Elvis_Review,Tested,NaN,NaN,0,1
29,2023-09-22 00:00:00,5493,HereAPI,Kirubakaran,Use,,,Elvis_Review,Elvis_Review,HOME ADDRESS?,...,NaN,Fixable - needs additional review,HOME ADDRESS?,Elvis_Review,Elvis_Review,Tested,NaN,NaN,0,1
42,2023-09-22 00:00:00,5523,HereAPI,HereAPI,use,HereAPI Approved,,Approved,Approved,NaN,...,NaN,Delete,NaN,Approved,Approved,Tested,NaN,NaN,0,1
68,2023-09-22 00:00:00,5565,HereAPI,HereAPI,use,HereAPI Approved,,Approved,Approved,NaN,...,NaN,NaN,NaN,Approved,Approved,Tested,NaN,NaN,0,1
121,2023-09-22 00:00:00,5803,HereAPI,"Gowtham, Brian",Use,NaN,,Elvis_Review,Elvis_Review,B transfer = Miami-Dade Transit (MDT),...,NaN,NaN,NaN,Elvis_Review,Elvis_Review,Tested,NaN,NaN,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6119,2023-10-28 00:00:00,21188,HereAPI,"Saurav, Brian",Use,NaN,NaN,Elvis_Review,Elvis_Review,B transfer = Miami-Dade Transit (MDT),...,NaN,NaN,NaN,Review,Review,Tested,NaN,NaN,1,1
6323,2023-10-28 00:00:00,21570,HereAPI,Himanshu,Use,NaN,NaN,Elvis_Review,Elvis_Review,NaN,...,NaN,Delete,NaN,Review,Review,Tested,NaN,NaN,0,1
6456,2023-10-28 00:00:00,21819,HereAPI,Sakshi,Use,NaN,NaN,Approved,Elvis_Review,NaN,...,NaN,NaN,NaN,Approved,Review,Tested,NaN,NaN,0,1
6589,2023-10-28 00:00:00,22096,HereAPI,"T A Divya, Brian",Use,NaN,NaN,Elvis_Review,Elvis_Review,Add B transfer Palm Tran,...,NaN,NaN,NaN,Review,Review,Tested,NaN,NaN,0,1


In [17]:
merged_df = pd.merge(merged_df, transfer_df['id'], on='id', how='left', indicator=True)

# Create a new column 'Traditional_Check' based on the merge indicator
merged_df['Transfer_Distance_Check'] = (merged_df['_merge'] == 'both').astype(int)

# Drop the indicator column and display the resulting DataFrame
merged_df = merged_df.drop(columns=['_merge'])

In [18]:
merged_df[merged_df['Transfer_Distance_Check']==1]

,Elvis_Date,elvis_id,1st Cleaner,FINAL_REVIEWER,Final_Usage,REASON FOR REMOVAL,REASON FOR REMOVAL [Other],route_match_flag,distance_flag,SUPERVISOR_COMMENT,...,ELVIS_STATUS,ELVIS_COMMENT,ROUTE_STATUS,Stops_Status,Test_Status,ROUTE_ONLY,Unnamed: 30,Traditional_Check,OD_Distance_Check,Transfer_Distance_Check
59,2023-09-22 00:00:00,5552,HereAPI,Kirubakaran,Use,,,Elvis_Review,Elvis_Review,NaN,...,Delete,NaN,Elvis_Review,Elvis_Review,Tested,NaN,NaN,1,0,1
210,2023-09-22 00:00:00,6154,HereAPI,Vaishnavi,Use,,,Elvis_Review,Elvis_Review,"D=12801 West Sunrise Boulevard, Sunrise, FL, USA",...,Fixable - needs additional review,"D=12801 West Sunrise Boulevard, Sunrise, FL, USA",Elvis_Review,Elvis_Review,Tested,NaN,NaN,1,0,1
222,2023-09-22 00:00:00,6211,HereAPI,"Vaishnavi, Brian",Use,NaN,,Elvis_Review,Elvis_Review,Delate B transfer. Add A transfer Route 7.,...,NaN,NaN,Elvis_Review,Elvis_Review,Tested,NaN,NaN,0,0,1
280,2023-09-30 00:00:00,6424,HereAPI,Priyanka,Use,,,Elvis_Review,Elvis_Review,NaN,...,NaN,NaN,Elvis_Review,Elvis_Review,Tested,NaN,NaN,0,0,1
316,2023-09-30 00:00:00,6550,HereAPI,Priyanka,Use,,,Elvis_Review,Elvis_Review,NaN,...,NaN,NaN,Elvis_Review,Elvis_Review,Tested,NaN,NaN,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6583,2023-10-28 00:00:00,22071,HereAPI,Bhumika,Use,NaN,NaN,Elvis_Review,Elvis_Review,NaN,...,NaN,NaN,Review,Review,Tested,NaN,NaN,1,0,1
6595,2023-10-28 00:00:00,22113,HereAPI,Himanshu,Use,NaN,NaN,Approved,Elvis_Review,NaN,...,NaN,NaN,Approved,Review,Tested,NaN,NaN,0,0,1
6613,2023-10-28 00:00:00,22161,HereAPI,"Kashish, Brian",Use,NaN,NaN,Elvis_Review,Elvis_Review,Nf1 = COMMERCIAL B/WOODLANDS B,...,NaN,NaN,Review,Review,Tested,NaN,NaN,0,0,1
6616,2023-10-28 00:00:00,22170,HereAPI,Himanshu,Use,NaN,NaN,Elvis_Review,Elvis_Review,NaN,...,NaN,NaN,Review,Review,Tested,NaN,NaN,0,0,1


In [19]:
merged_df.drop(columns=['Unnamed: 30'],inplace=True)

In [20]:
merged_df

,Elvis_Date,elvis_id,1st Cleaner,FINAL_REVIEWER,Final_Usage,REASON FOR REMOVAL,REASON FOR REMOVAL [Other],route_match_flag,distance_flag,SUPERVISOR_COMMENT,...,_INTRV_NOTE,ELVIS_STATUS,ELVIS_COMMENT,ROUTE_STATUS,Stops_Status,Test_Status,ROUTE_ONLY,Traditional_Check,OD_Distance_Check,Transfer_Distance_Check
0,2023-09-22 00:00:00,4981,HereAPI,Kirubakaran,Use,,,Elvis_Review,Elvis_Review,NaN,...,NaN,NaN,NaN,Elvis_Review,Elvis_Review,Tested,NaN,0,0,0
1,2023-09-22 00:00:00,4983,HereAPI,Kirubakaran,Use,,,Elvis_Review,Elvis_Review,NaN,...,NaN,NaN,NaN,Elvis_Review,Elvis_Review,Tested,NaN,0,0,0
2,2023-09-22 00:00:00,4989,HereAPI,Kirubakaran,Use,,,Elvis_Review,Elvis_Review,NaN,...,NaN,NaN,NaN,Elvis_Review,Elvis_Review,Tested,NaN,0,0,0
3,2023-09-22 00:00:00,5011,HereAPI,Kirubakaran,Use,,,Elvis_Review,Elvis_Review,NaN,...,NaN,NaN,NaN,Elvis_Review,Elvis_Review,Tested,NaN,0,0,0
4,2023-09-22 00:00:00,5061,HereAPI,Kirubakaran,Use,,,Elvis_Review,Elvis_Review,NaN,...,NaN,NaN,NaN,Elvis_Review,Elvis_Review,Tested,NaN,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6615,2023-10-28 00:00:00,22163,HereAPI,Himanshu,Use,NaN,NaN,Elvis_Review,Elvis_Review,NaN,...,NaN,NaN,NaN,Review,Review,Tested,NaN,0,0,0
6616,2023-10-28 00:00:00,22170,HereAPI,Himanshu,Use,NaN,NaN,Elvis_Review,Elvis_Review,NaN,...,NaN,NaN,NaN,Review,Review,Tested,NaN,0,0,1
6617,2023-10-28 00:00:00,22171,HereAPI,"Kashish, Brian",Use,NaN,NaN,Elvis_Review,Elvis_Review,NaN,...,NaN,NaN,NaN,Elvis_Review,Elvis_Review,Tested,NaN,0,0,0
6618,2023-10-28 00:00:00,22172,HereAPI,"Kashish, Brian",Use,NaN,NaN,Approved,Elvis_Review,NaN,...,NaN,Approved - looks wrong but confirmed correct,test,Approved,Review,Tested,NaN,0,0,1


In [21]:
merged_df[(merged_df['Transfer_Distance_Check']==1)  | (merged_df['OD_Distance_Check']==1) | (merged_df['Traditional_Check']==1)]

,Elvis_Date,elvis_id,1st Cleaner,FINAL_REVIEWER,Final_Usage,REASON FOR REMOVAL,REASON FOR REMOVAL [Other],route_match_flag,distance_flag,SUPERVISOR_COMMENT,...,_INTRV_NOTE,ELVIS_STATUS,ELVIS_COMMENT,ROUTE_STATUS,Stops_Status,Test_Status,ROUTE_ONLY,Traditional_Check,OD_Distance_Check,Transfer_Distance_Check
18,2023-09-22 00:00:00,5454,HereAPI,Kirubakaran,Use,,,Elvis_Review,Elvis_Review,"1015 North America Way, Miami, FL, USA, REMOVE...",...,NaN,Fixable - needs additional review,"1015 North America Way, Miami, FL, USA, REMOVE...",Elvis_Review,Elvis_Review,Tested,NaN,0,1,0
29,2023-09-22 00:00:00,5493,HereAPI,Kirubakaran,Use,,,Elvis_Review,Elvis_Review,HOME ADDRESS?,...,NaN,Fixable - needs additional review,HOME ADDRESS?,Elvis_Review,Elvis_Review,Tested,NaN,0,1,0
42,2023-09-22 00:00:00,5523,HereAPI,HereAPI,use,HereAPI Approved,,Approved,Approved,NaN,...,NaN,Delete,NaN,Approved,Approved,Tested,NaN,0,1,0
59,2023-09-22 00:00:00,5552,HereAPI,Kirubakaran,Use,,,Elvis_Review,Elvis_Review,NaN,...,NaN,Delete,NaN,Elvis_Review,Elvis_Review,Tested,NaN,1,0,1
68,2023-09-22 00:00:00,5565,HereAPI,HereAPI,use,HereAPI Approved,,Approved,Approved,NaN,...,NaN,NaN,NaN,Approved,Approved,Tested,NaN,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6595,2023-10-28 00:00:00,22113,HereAPI,Himanshu,Use,NaN,NaN,Approved,Elvis_Review,NaN,...,NaN,NaN,NaN,Approved,Review,Tested,NaN,0,0,1
6612,2023-10-28 00:00:00,22160,HereAPI,Himanshu,Use,NaN,NaN,Elvis_Review,Elvis_Review,NaN,...,NaN,NaN,NaN,Review,Review,Tested,NaN,0,1,0
6613,2023-10-28 00:00:00,22161,HereAPI,"Kashish, Brian",Use,NaN,NaN,Elvis_Review,Elvis_Review,Nf1 = COMMERCIAL B/WOODLANDS B,...,NaN,NaN,NaN,Review,Review,Tested,NaN,0,0,1
6616,2023-10-28 00:00:00,22170,HereAPI,Himanshu,Use,NaN,NaN,Elvis_Review,Elvis_Review,NaN,...,NaN,NaN,NaN,Review,Review,Tested,NaN,0,0,1
